In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import base64

from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [3]:
df = pd.read_csv("spam_Emails_data.csv")
print(df.head())

  label                                               text
0  Spam  viiiiiiagraaaa\nonly for the ones that want to...
1   Ham  got ice thought look az original message ice o...
2  Spam  yo ur wom an ne eds an escapenumber in ch ma n...
3  Spam  start increasing your odds of success & live s...
4   Ham  author jra date escapenumber escapenumber esca...


In [4]:
stopWordList = [
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 'and',
    'any', 'are', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being',
    'below', 'between', 'both', 'but', 'by', "can't", 'cannot', 'could', "couldn't",
    'did', "didn't", 'do', 'does', "doesn't", 'doing', "don't", 'down', 'during',
    'each', 'few', 'for', 'from', 'further', 'had', "hadn't", 'has', "hasn't",
    'have', "haven't", 'having', 'he', "he'd", "he'll", "he's", 'her', 'here',
    "here's", 'hers', 'herself', 'him', 'himself', 'his', 'how', "how's", 'i',
    "i'd", "i'll", "i'm", "i've", 'if', 'in', 'into', 'is', "isn't", 'it', "it's",
    'its', 'itself', "let's", 'me', 'more', 'most', "mustn't", 'my', 'myself', 'no',
    'nor', 'not', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'ought', 'our',
    'ours', 'ourselves', 'out', 'over', 'own', 'same', "shan't", 'she', "she'd",
    "she'll", "she's", 'should', "shouldn't", 'so', 'some', 'such', 'than', 'that',
    "that's", 'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there',
    "there's", 'these', 'they', "they'd", "they'll", "they're", "they've", 'this',
    'those', 'through', 'to', 'too', 'under', 'until', 'up', 'very', 'was',
    "wasn't", 'we', "we'd", "we'll", "we're", "we've", 'were', "weren't", 'what',
    "what's", 'when', "when's", 'where', "where's", 'which', 'while', 'who',
    "who's", 'whom', 'why', "why's", 'with', "won't", 'would', "wouldn't", 'you',
    "you'd", "you'll", "you're", "you've", 'your', 'yours', 'yourself', 'yourselves',
    "of"
]

stopWordListCleaned = [w.replace("'", "") for w in stopWordList]

#x_text = df["text"]
#y = df["label"].map({'Ham': 0, 'Spam': 1})

df = df.dropna(subset = ["text"])

x_train_text, x_test_text, y_train, y_test = train_test_split(
    df["text"], 
    df["label"].map({'Ham': 0, 'Spam': 1}), 
    test_size = 0.2, 
    random_state = 42
)



vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    min_df = 10,
    max_df = 0.95,
    stop_words = stopWordListCleaned
)

x_train = vectorizer.fit_transform(x_train_text)
x_test = vectorizer.transform(x_test_text)

model = LogisticRegression(max_iter = 10000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98     20317
           1       0.98      0.99      0.98     18453

    accuracy                           0.98     38770
   macro avg       0.98      0.98      0.98     38770
weighted avg       0.98      0.98      0.98     38770



In [5]:
# Get feature names from the vectorizer
feature_names = vectorizer.get_feature_names_out()

# Get the coefficients for the positive class (spam)
coefs = model.coef_[0]

# Pair each feature with its coefficient
feature_weights = list(zip(feature_names, coefs))

# Sort by coefficient descending (largest positive = strongest spam indicator)
feature_weights_sorted = sorted(feature_weights, key=lambda x: x[1], reverse=True)

# Print top 20 features
top_20 = feature_weights_sorted[:20]
for feature, weight in top_20:
    print(feature, weight)

http 7.423431100786091
php 5.6918320614801585
money 5.224133581834169
escapenumber email 4.854152097608548
quality 4.776316042499129
info 4.69007741082088
save 4.521399699897885
now 4.51920592400607
hk 4.478284171861736
remove 4.471451747728713
escapelong 4.2458679727108075
life 4.206360233098656
bescapenumber 4.115930176997446
offer 4.099425044847202
reply 4.010581493510558
net 3.96279631114443
price escapenumber 3.9527823273785425
pills 3.866460717198289
anatrim 3.782710415712043
2004 3.7628679443915636


In [6]:
test_predictions = [
    "Congratulations! You won a free iPhone",
    "Hey, are we still meeting tomorrow?",
    "You’ve been selected for a $1000 Amazon gift card. Claim now!",
    "Don’t forget to bring the report to the meeting today.",
    "Urgent! Your account has been compromised. Verify immediately.",
    "Happy Birthday! Hope you have a wonderful day!",
    "Limited time offer! 90% off on all electronics. Shop now!",
    "Can you send me the notes from class yesterday?",
    "You have a new voicemail. Listen here: [link]",
    "Lunch at 12? Let me know if that works.",
    "Winner! You have won a free cruise vacation. Click to claim.",
    "Are we still on for the gym later?",
    "Your loan is approved! Transfer funds within 24 hours.",
    "Meeting rescheduled to 3 PM, please confirm.",
    "Exclusive deal: Get free Bitcoin today. Don’t miss out!",
    "Thanks for helping me with my homework!",
    "Congratulations! You’ve been chosen for a free prize. Act now!",
    "What time are you picking me up tomorrow?",
    "Alert: Suspicious activity detected on your account. Verify now.",
    "Can you review the presentation before tomorrow’s meeting?"
]

x_input = vectorizer.transform(test_predictions)

preds = model.predict(x_input)

print(preds)

probs = model.predict_proba(x_input)

for i in range (len(test_predictions)):
    confidence = probs[i][1]
    
    if preds[i] == 1:
        label = "Spam" 

        print(f"Email: {test_predictions[i]}")
        print(f"Prediction: {label}")
        print(f"Spam Confidence: {confidence:.2f}")
        print()
    
    else:
        
        label = "Real"
        
        print(f"Email: {test_predictions[i]}")
        print(f"Prediction: {label}")
        print(f"Real Confidence: {1 - confidence:.2f}")
        print()
    
    

[1 0 1 0 1 0 1 0 1 0 1 1 1 0 1 0 1 0 1 0]
Email: Congratulations! You won a free iPhone
Prediction: Spam
Spam Confidence: 0.76

Email: Hey, are we still meeting tomorrow?
Prediction: Real
Real Confidence: 0.97

Email: You’ve been selected for a $1000 Amazon gift card. Claim now!
Prediction: Spam
Spam Confidence: 0.98

Email: Don’t forget to bring the report to the meeting today.
Prediction: Real
Real Confidence: 0.92

Email: Urgent! Your account has been compromised. Verify immediately.
Prediction: Spam
Spam Confidence: 0.95

Email: Happy Birthday! Hope you have a wonderful day!
Prediction: Real
Real Confidence: 0.75

Email: Limited time offer! 90% off on all electronics. Shop now!
Prediction: Spam
Spam Confidence: 0.99

Email: Can you send me the notes from class yesterday?
Prediction: Real
Real Confidence: 0.75

Email: You have a new voicemail. Listen here: [link]
Prediction: Spam
Spam Confidence: 0.72

Email: Lunch at 12? Let me know if that works.
Prediction: Real
Real Confidence: 

In [7]:
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

flow = InstalledAppFlow.from_client_secrets_file(
    'credentials.json', SCOPES)

creds = flow.run_local_server(port=0)

service = build('gmail', 'v1', credentials=creds)

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=759067055974-klebnofra38um7al5slvfk9tv9ai82qa.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A57902%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=gYn69TqCfv1DWkebwGLQyHYykGYMZ2&code_challenge=ApzWIrZMahPcv1TpXGMS-44lRYMPEZUKlnLj-7bCAyo&code_challenge_method=S256&access_type=offline


In [8]:
results = service.users().messages().list(userId='me', maxResults=10).execute()
messages = results.get('messages', [])

In [9]:
def get_email_body(msg):
    payload = msg['payload']
    parts = payload.get('parts', [])

    for part in parts:
        if part['mimeType'] == 'text/plain':
            data = part['body']['data']
            return base64.urlsafe_b64decode(data).decode()

    return ""

In [26]:
emails = []

for m in messages:
    msg = service.users().messages().get(userId='me', id=m['id']).execute()
    text = get_email_body(msg)
    emails.append(text)
    
    print(text)
    print()
    print()
    print()
    print("--------------------------------------------------------------")
    print("NEW EMAIL")
    print()
    print()
    print()

Hi Nathan,

Happy Thursday! As AI advances, one question every student will face is: how will it affect jobs? In 2026, we built a project that helps students explore this for themselves.

In AI and Future of Work, developed by Drew Solomon, Brown University, students analyze real labor market data to figure out which jobs are most at risk from automation, and walk away with a working web app and a policy brief on reskilling for the future. Find our full 2026 project lineup here<https://www.inspiritai.com/project#projectoptions>.

Apply here<https://inspiritai.paperform.co/?utm_campaign=0322alma> by April 15th or reach us at +1 650 677 8686 with any questions.

Warmly,
Varsha Sandadi


If you want to opt out, kindly click here<https://replysof.com/home/index?seqId=1648507&spId=526926566&stepId=7005747>.






--------------------------------------------------------------
NEW EMAIL



Do you know Gabrielle Lenchner?5 mutual connections

Yes, connect: https://www.linkedin.com/comm/mynetwo

In [41]:
def get_email_subject(msg):
    headers = msg['payload']['headers']
    
    for header in headers:
        if header['name'] == 'Subject':
            return header['value']
    
    return "No Subject"

def get_email_sender(msg):
    headers = msg['payload']['headers']
    
    for header in headers:
        if header['name'] == 'From':
            return header['value']
    
    return "No Subject"

x_input = vectorizer.transform(emails)

preds_new = model.predict(x_input)
probs_new = model.predict_proba(x_input)
print(preds_new)
print()

for i in range(len(messages)):
    
    msg = service.users().messages().get(
    userId='me',
    id=messages[i]['id'],
    format='full'
    ).execute()
    
    email_text = get_email_body(msg)
    sender = get_email_sender(msg)
    subject = get_email_subject(msg)

    confidence = probs_new[i][1]

    preview = " ".join(email_text.split()[:10])

    print(f"From: {sender}")
    print(f"Subject: {subject}")
    print(f"Preview: {preview}...")

    if preds_new[i] == 1:
        print(f"Prediction: Spam")
        print(f"Spam Confidence: {confidence:.2f}")
    else:
        print(f"Prediction: Real")
        print(f"Real Confidence: {1 - confidence:.2f}")

    print()

[0 0 1 1 1 1 1 1 1 1]

From: Varsha Sandadi <varshas@inspiritai-hs.com>
Subject: Explore the Future of Work with Inspirit AI
Preview: Hi Nathan, Happy Thursday! As AI advances, one question every...
Prediction: Real
Real Confidence: 0.64

From: LinkedIn <messages-noreply@linkedin.com>
Subject: Nathan, add Gabrielle Lenchner - Sales Floor Team Member 💬
Preview: Do you know Gabrielle Lenchner?5 mutual connections Yes, connect: https://www.linkedin.com/comm/mynetwork/send-invite/gabrielle-lenchner-569999315/?lipi=urn%3Ali%3Apage%3Aemail_email_pymk_02%3BQVG7e0JXQBOutd70z6Z1hw%3D%3D&midSig=0J1sJeoSz-BYc1&midToken=AQG-vc8_6VA-7Q&trkEmail=eml-email_pymk_02-pymkCard-0-pymk_cta%3A+urn%3Ali%3Amember%3A1343149281-null-q3cuj4%7Emnrh1h38%7E54-null-null&trk=eml-email_pymk_02-pymkCard-0-pymk_cta%3A+urn%3Ali%3Amember%3A1343149281&_sig=1c-CmTuvz-BYc1...
Prediction: Real
Real Confidence: 0.53

From: Shutterfly <Shutterfly@em.shutterfly.com>
Subject: Time to make a photo book! 40% off your entire photo b